# AI-Powered Insider Threat Detection — Real-Library Training (Google Colab)

This notebook re-runs the project's model training and explainability using
the **actual libraries named in the proposal** — scikit-learn, TensorFlow,
SHAP, and LIME — rather than the from-scratch NumPy implementations built
during initial development (where the dev sandbox's network proxy could not
download these packages' larger dependencies; see `docs/day4_progress.md`,
`docs/day5_progress.md`, `docs/day6_progress.md` in the repo for the full story).

Colab has unrestricted internet, so all of these install normally here.

**What this notebook does:**
1. Installs scikit-learn, TensorFlow, SHAP, LIME
2. Pulls the already-generated dataset directly from the GitHub repo (no re-upload needed)
3. Trains real `IsolationForest`, `OneClassSVM`, and a Keras `Autoencoder`
4. Evaluates all three against the ground-truth labels (precision/recall/F1/ROC-AUC)
5. Runs real `shap` and `lime` explanations for the flagged users
6. Compares these results against the from-scratch versions already committed to the repo

**How to run:** `Runtime > Run all`. Takes a few minutes end to end.

## 1. Install real libraries

On Colab's normal internet this takes seconds, not the minutes-that-never-finish it took in the constrained dev sandbox.

In [ ]:
!pip install -q scikit-learn tensorflow shap lime networkx
print("done")

## 2. Load the dataset directly from GitHub

The dataset (100 simulated users, 30 days of logon/device/http/email/file
logs, 6 labeled insider scenarios, and the 25 engineered features from Day 3)
is already committed to the repo — no need to regenerate or re-upload it.

In [ ]:
import pandas as pd
import numpy as np

REPO_RAW = "https://raw.githubusercontent.com/Perera1325/ai-powered-insider-threat-detection/main/data/processed/full"

features = pd.read_csv(f"{REPO_RAW}/user_features.csv")
labels = pd.read_csv(f"{REPO_RAW}/labels.csv")

feature_cols = [c for c in features.columns if c != "user_id"]
merged = features.merge(labels[["user_id", "is_malicious", "scenario"]], on="user_id", how="left")
y_true = merged["is_malicious"].fillna(False).astype(int).values

print(features.shape, "users x features")
print(f"{y_true.sum()} known malicious users out of {len(y_true)}")
features.head()

## 3. Standardize features

Same standardization approach as the original pipeline (Day 4) — z-score
each feature so all 25 are on a comparable scale before training.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(features[feature_cols].values)
X.shape

## 4. Train real scikit-learn Isolation Forest + One-Class SVM

These are the exact two models named in the project proposal. `contamination`
/ `nu` are set to roughly match the known 6% base rate in this dataset — in a
real deployment you would not know this in advance and would need to pick a
reasonable estimate or tune it.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

iso = IsolationForest(n_estimators=300, contamination=0.06, random_state=42)
iso.fit(X)
# sklearn's score_samples: HIGHER = more normal, so flip sign for an anomaly score
iso_scores = -iso.score_samples(X)

ocsvm = OneClassSVM(kernel="rbf", nu=0.06, gamma="scale")
ocsvm.fit(X)
ocsvm_scores = -ocsvm.score_samples(X)

print("Isolation Forest score range:", iso_scores.min(), iso_scores.max())
print("One-Class SVM score range:", ocsvm_scores.min(), ocsvm_scores.max())

## 5. Train a real TensorFlow/Keras Autoencoder

Same architecture idea as the from-scratch version (Day 5): compress to a
small bottleneck, reconstruct, and use reconstruction error as the anomaly
score. Kept deliberately small (bottleneck=2, few epochs) — Day 5's write-up
found that an over-capacity/over-trained autoencoder learns to reconstruct
the anomalies too well, which defeats the purpose.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

tf.random.set_seed(42)

n_features = X.shape[1]
inputs = layers.Input(shape=(n_features,))
encoded = layers.Dense(16, activation="relu")(inputs)
bottleneck = layers.Dense(2, activation="relu")(encoded)
decoded = layers.Dense(16, activation="relu")(bottleneck)
outputs = layers.Dense(n_features, activation="linear")(decoded)

autoencoder = models.Model(inputs, outputs)
autoencoder.compile(optimizer="adam", loss="mse")
history = autoencoder.fit(X, X, epochs=200, batch_size=16, verbose=0)

reconstructions = autoencoder.predict(X, verbose=0)
ae_scores = np.mean((reconstructions - X) ** 2, axis=1)

print("Final training loss:", history.history["loss"][-1])
print("Autoencoder score range:", ae_scores.min(), ae_scores.max())

## 6. Evaluate all three against the ground truth

Same evaluation approach as Day 4: flag the top-K users by score (K = number
of known malicious users) and compute precision/recall/F1, plus ROC-AUC
across all thresholds.

In [ ]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

k = int(y_true.sum())

def evaluate(scores, name):
    top_k_idx = np.argsort(-scores)[:k]
    pred = np.zeros_like(y_true)
    pred[top_k_idx] = 1
    precision = precision_score(y_true, pred, zero_division=0)
    recall = recall_score(y_true, pred, zero_division=0)
    f1 = f1_score(y_true, pred, zero_division=0)
    auc = roc_auc_score(y_true, scores)
    return {"model": name, "precision@k": precision, "recall@k": recall, "f1@k": f1, "roc_auc": auc}

results = pd.DataFrame([
    evaluate(iso_scores, "IsolationForest (sklearn)"),
    evaluate(ocsvm_scores, "OneClassSVM (sklearn)"),
    evaluate(ae_scores, "Autoencoder (TensorFlow/Keras)"),
]).set_index("model")

results.round(3)

## 7. Compare against the from-scratch results already in the repo

This is the key comparison: do the real, industry-standard libraries agree
with the from-scratch implementations built during initial development?

In [ ]:
scratch_results = pd.read_csv(f"{REPO_RAW}/day4_model_comparison.csv", index_col="model")
print("From-scratch results (already in the repo, Day 4/5):")
display(scratch_results.round(3))

print("\nReal-library results (this notebook, on Colab):")
display(results.round(3))

## 8. Real SHAP explanations

Uses `shap.KernelExplainer` (model-agnostic, works with any `.decision_function`-
style scorer) against the Isolation Forest — same setup as the from-scratch
Kernel SHAP in Day 6, but using the reference implementation.

In [ ]:
import shap

background = shap.sample(X, 30, random_state=42)
explainer = shap.KernelExplainer(lambda x: -iso.score_samples(x), background)

malicious_mask = y_true == 1
malicious_idx = np.where(malicious_mask)[0]

shap_values = explainer.shap_values(X[malicious_idx], nsamples=200)

for i, idx in enumerate(malicious_idx):
    uid = features.iloc[idx].user_id
    scenario = merged.iloc[idx].scenario
    top_features = pd.Series(shap_values[i], index=feature_cols).sort_values(key=abs, ascending=False).head(5)
    print(f"\n{uid} (scenario: {scenario})")
    print(top_features.round(3).to_string())

## 9. Real LIME explanation (one example)

Uses `lime.lime_tabular.LimeTabularExplainer`, the reference implementation,
against the same Isolation Forest scorer.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_explainer = LimeTabularExplainer(
    X, feature_names=feature_cols, mode="regression", random_state=42
)

example_idx = malicious_idx[0]
example_uid = features.iloc[example_idx].user_id
exp = lime_explainer.explain_instance(
    X[example_idx], lambda x: -iso.score_samples(x), num_features=8
)

print(f"LIME explanation for {example_uid}:")
for feature, weight in exp.as_list():
    print(f"  {feature}: {weight:.4f}")

## 10. Save results back out

Download these and commit them into `data/processed/full/` in your repo
(alongside the from-scratch versions) if you want both sets of results
available for the final report.

In [ ]:
results.to_csv("real_library_model_comparison.csv")
from google.colab import files
files.download("real_library_model_comparison.csv")

## Notes for the final report

- If the real-library results closely match the from-scratch results
  (they should — both implement the same underlying algorithms), that's
  good evidence the from-scratch versions built during initial development
  were correct, not just lucky.
- Any differences are worth discussing briefly: scikit-learn's Isolation
  Forest and One-Class SVM have additional internal optimizations (e.g.
  different splitting/kernel numerics) that can shift scores slightly
  without changing the overall ranking.
- The Streamlit dashboard (`app.py` in the repo) is not run from this
  notebook — Colab doesn't serve local web apps the same way a normal
  machine does. Run `streamlit run app.py` on your own computer (or VS
  Code's terminal) as covered earlier; it only needed `pip install -r
  requirements.txt`, which works fine outside this constrained sandbox.